# Unsupervised Learning 🔍

What if you don't have labels? Unsupervised learning finds patterns in unlabeled data. Clustering, dimensionality reduction, anomaly detection.

## What You'll Learn
- Clustering algorithms (K-Means, DBSCAN, Hierarchical)
- Dimensionality reduction (PCA, t-SNE)
- When to use each method
- Evaluating clusters (silhouette score, elbow method)
- Real-world applications
- Finding optimal number of clusters

## Supervised vs Unsupervised Learning 🏷️

### Supervised Learning (Labeled Data)
```
Input: X = [features]      Output: y = [labels]
        [height, weight]           ["overweight"]
        [height, weight]           ["normal"]
        [height, weight]           ["underweight"]

Goal: Learn mapping X → y
Example: Classification, Regression
```

### Unsupervised Learning (Unlabeled Data)
```
Input: X = [features]      Output: (patterns discovered)
        [height, weight]
        [height, weight]
        [height, weight]

Goal: Find structure/patterns
Example: Clustering, Dimensionality Reduction
```

### Real-World Applications

**Clustering:**
- Customer segmentation (groups of similar customers)
- Document clustering (group similar documents)
- Gene sequencing (group similar sequences)
- Anomaly detection (find unusual patterns)

**Dimensionality Reduction:**
- Visualization (2D/3D from 100+ dimensions)
- Feature compression (reduce data size)
- Noise removal (filter out noise)
- Speeding up supervised learning

## Clustering: K-Means 🎯

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import seaborn as sns

print("📚 Libraries loaded!")

In [ ]:
# Create simple 2D clustering dataset
X, y_true = make_blobs(n_samples=300, centers=3, n_features=2, 
                        cluster_std=0.8, random_state=42)

plt.figure(figsize=(10, 6))
plt.scatter(X[:, 0], X[:, 1], alpha=0.6, s=100, edgecolors='black', linewidth=0.5)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Unlabeled Data: Can you spot the clusters?')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Notice: Points seem to form 3 natural groups!")

In [ ]:
# K-Means with k=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
y_pred = kmeans.fit_predict(X)

print("="*60)
print("K-MEANS CLUSTERING")
print("="*60)

print(f"\nNumber of clusters: 3")
print(f"Cluster assignments: {np.bincount(y_pred)}")
print(f"Inertia (within-cluster sum of squares): {kmeans.inertia_:.2f}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original data
axes[0].scatter(X[:, 0], X[:, 1], alpha=0.6, s=100, edgecolors='black', linewidth=0.5)
axes[0].set_title('Original Data (Unlabeled)')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')
axes[0].grid(True, alpha=0.3)

# Clustered data
colors = ['red', 'blue', 'green']
for i in range(3):
    cluster_points = X[y_pred == i]
    axes[1].scatter(cluster_points[:, 0], cluster_points[:, 1], 
                   c=colors[i], label=f'Cluster {i}', s=100, alpha=0.6, edgecolors='black', linewidth=0.5)

# Plot cluster centers
axes[1].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
               marker='X', s=300, c='yellow', edgecolors='black', linewidth=2, label='Centers')

axes[1].set_title('K-Means Clustering (k=3)')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ K-Means found 3 clusters!")
print("✅ Yellow X marks are cluster centers")

In [ ]:
# How K-Means works (simplified)
print("\n" + "="*60)
print("HOW K-MEANS WORKS")
print("="*60)

explanation = """
Algorithm Steps:

1️⃣  INITIALIZE: Place k random centers in the data space

2️⃣  ASSIGN: Assign each point to nearest center
            (creates k clusters)

3️⃣  UPDATE: Move centers to mean of their cluster points
            (new center positions)

4️⃣  REPEAT: Go back to step 2 until centers don't move
            (convergence)

Result: Final cluster assignments

Time Complexity: O(n·k·i·d)
  n = number of points
  k = number of clusters
  i = iterations
  d = dimensions

✅ Pros:
  • Fast (scales to large datasets)
  • Simple to understand and implement
  • Works well with spherical clusters

❌ Cons:
  • Must specify k in advance
  • Sensitive to initialization (local optima)
  • Struggles with non-spherical clusters
  • Assumes clusters of similar size
"""

print(explanation)

In [ ]:
# Finding optimal number of clusters - Elbow Method
print("\n" + "="*60)
print("FINDING OPTIMAL K: ELBOW METHOD")
print("="*60)

inertias = []
silhouette_scores = []
K_range = range(1, 10)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X)
    inertias.append(kmeans.inertia_)
    
    if k > 1:  # Silhouette score needs at least 2 clusters
        score = silhouette_score(X, labels)
        silhouette_scores.append(score)
    else:
        silhouette_scores.append(None)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow plot
axes[0].plot(K_range, inertias, marker='o', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia (Within-cluster sum of squares)')
axes[0].set_title('Elbow Method: Find the "Elbow"')
axes[0].grid(True, alpha=0.3)
axes[0].axvline(x=3, color='red', linestyle='--', alpha=0.5, label='Optimal k=3')
axes[0].legend()

# Silhouette plot
silhouette_scores_valid = [s for s in silhouette_scores if s is not None]
axes[1].plot(range(2, 10), silhouette_scores_valid, marker='s', linewidth=2, markersize=8, color='orange')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score: Higher is Better')
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(range(2, 10))
optimal_k = np.argmax(silhouette_scores_valid) + 2
axes[1].axvline(x=optimal_k, color='red', linestyle='--', alpha=0.5, label=f'Optimal k={optimal_k}')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nElbow Method:")
print(f"  Look for 'elbow' point (sharp change in slope)")
print(f"  Here: k=3 seems optimal")

print(f"\nSilhouette Score:")
print(f"  Range: -1 to 1 (higher is better)")
print(f"  1 = perfect clustering")
print(f"  0 = overlapping clusters")
print(f" -1 = wrong cluster assignment")
print(f"  Optimal k={optimal_k}")

## DBSCAN: Density-Based Clustering 🌐

In [ ]:
# Create non-spherical dataset
X_moons, y_true_moons = make_moons(n_samples=300, noise=0.05, random_state=42)

# Compare K-Means vs DBSCAN
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Original
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], alpha=0.6, s=100, edgecolors='black', linewidth=0.5)
axes[0].set_title('Original Data (Moons Dataset)')
axes[0].grid(True, alpha=0.3)

# K-Means (struggles)
kmeans = KMeans(n_clusters=2, random_state=42)
y_kmeans = kmeans.fit_predict(X_moons)
colors = ['red', 'blue']
for i in range(2):
    axes[1].scatter(X_moons[y_kmeans == i, 0], X_moons[y_kmeans == i, 1],
                   c=colors[i], label=f'Cluster {i}', s=100, alpha=0.6, edgecolors='black', linewidth=0.5)
axes[1].set_title('K-Means (Fails on Non-Spherical)')
axes[1].grid(True, alpha=0.3)

# DBSCAN (works better)
dbscan = DBSCAN(eps=0.1, min_samples=5)
y_dbscan = dbscan.fit_predict(X_moons)
n_clusters = len(set(y_dbscan)) - (1 if -1 in y_dbscan else 0)
n_noise = list(y_dbscan).count(-1)

colors_dbscan = ['red', 'blue']
for i in range(n_clusters):
    axes[2].scatter(X_moons[y_dbscan == i, 0], X_moons[y_dbscan == i, 1],
                   c=colors_dbscan[i], label=f'Cluster {i}', s=100, alpha=0.6, edgecolors='black', linewidth=0.5)
if n_noise > 0:
    axes[2].scatter(X_moons[y_dbscan == -1, 0], X_moons[y_dbscan == -1, 1],
                   c='gray', marker='X', s=200, label='Noise', edgecolors='black', linewidth=0.5)
axes[2].set_title(f'DBSCAN (Works Better! Found {n_clusters} clusters, {n_noise} noise points)')
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ DBSCAN handles non-spherical clusters!")
print("✅ Finds noise points automatically")

In [ ]:
# DBSCAN explanation
print("="*60)
print("DBSCAN: DENSITY-BASED SPATIAL CLUSTERING")
print("="*60)

explanation = """
Core Idea: Points in dense regions are in same cluster

Parameters:
  • eps: Maximum distance between points in neighborhood
  • min_samples: Minimum points in eps-neighborhood to form core point

Algorithm:
1. For each point, count neighbors within eps
2. Points with >= min_samples neighbors = core points
3. Core points connect to form clusters
4. Non-core points near clusters = border points
5. Isolated points = noise

✅ Pros:
  • Finds arbitrary-shaped clusters
  • Detects outliers/noise automatically
  • Don't need to specify number of clusters
  • Works with high-dimensional data

❌ Cons:
  • Sensitive to eps and min_samples parameters
  • Struggles with varying cluster densities
  • Slower than K-Means for large datasets
"""

print(explanation)

## Dimensionality Reduction: PCA 📉

In [ ]:
# Load high-dimensional dataset
from sklearn.datasets import load_digits

digits = load_digits()
X_digits = digits.data
y_digits = digits.target

print("="*60)
print("DIMENSIONALITY REDUCTION: PCA")
print("="*60)

print(f"\nDigits Dataset:")
print(f"  Original shape: {X_digits.shape}")
print(f"  {X_digits.shape[0]} samples (handwritten digits 0-9)")
print(f"  {X_digits.shape[1]} features (8x8 pixel intensities)")

# Visualize some digits
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    digit_image = X_digits[i].reshape(8, 8)
    ax.imshow(digit_image, cmap='gray')
    ax.set_title(f'Digit: {y_digits[i]}')
    ax.axis('off')

plt.suptitle('Sample Handwritten Digits')
plt.tight_layout()
plt.show()

In [ ]:
# Apply PCA with different numbers of components
print("\nApplying PCA...")

# Full PCA to see variance explained
pca_full = PCA()
pca_full.fit(X_digits)

# Cumulative explained variance
cumsum_var = np.cumsum(pca_full.explained_variance_ratio_)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Variance explained
axes[0].plot(range(1, len(cumsum_var) + 1), cumsum_var, marker='o', linewidth=2)
axes[0].axhline(y=0.95, color='red', linestyle='--', alpha=0.5, label='95% explained')
axes[0].axvline(x=30, color='red', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Number of Components')
axes[0].set_ylabel('Cumulative Explained Variance')
axes[0].set_title('PCA: Cumulative Variance Explained')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Individual component variance
axes[1].bar(range(1, 31), pca_full.explained_variance_ratio_[:30], alpha=0.7)
axes[1].set_xlabel('Principal Component')
axes[1].set_ylabel('Variance Explained')
axes[1].set_title('Individual Component Variance')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Find how many components for 95% variance
n_components_95 = np.argmax(cumsum_var >= 0.95) + 1
print(f"\n✅ Need {n_components_95} components to explain 95% variance")
print(f"   Original: 64 dimensions")
print(f"   Reduced: {n_components_95} dimensions")
print(f"   Reduction: {(1 - n_components_95/64)*100:.0f}%")

In [ ]:
# Reconstruct digits with fewer components
n_components_list = [2, 10, 30, 64]

fig, axes = plt.subplots(len(n_components_list), 10, figsize=(15, 8))

for row, n_comp in enumerate(n_components_list):
    pca = PCA(n_components=n_comp)
    X_reduced = pca.fit_transform(X_digits)
    X_reconstructed = pca.inverse_transform(X_reduced)
    
    for col in range(10):
        digit_image = X_reconstructed[col].reshape(8, 8)
        axes[row, col].imshow(digit_image, cmap='gray')
        axes[row, col].axis('off')
    
    axes[row, 0].set_ylabel(f'{n_comp} components', fontsize=11, fontweight='bold')

plt.suptitle('Reconstruction Quality vs Number of Components', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Notice: With just 30 components (~95% variance), digits are very recognizable!")
print("       With 2 components, digits are unrecognizable but capture overall structure.")

In [ ]:
# PCA for visualization
print("\n" + "="*60)
print("PCA FOR 2D VISUALIZATION")
print("="*60)

pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_digits)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y_digits, 
                     cmap='tab10', s=50, alpha=0.6, edgecolors='black', linewidth=0.5)
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} variance)')
plt.title('PCA 2D Visualization: Handwritten Digits')
plt.colorbar(scatter, label='Digit')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n✅ Reduced from 64D to 2D")
print(f"✅ Explains {pca_2d.explained_variance_ratio_[0] + pca_2d.explained_variance_ratio_[1]:.1%} of variance")
print(f"✅ Can see clusters of similar digits!")

## PCA Explanation 🎨

In [ ]:
explanation = """
PRINCIPAL COMPONENT ANALYSIS (PCA)

Core Idea: Find directions of maximum variance in data

Steps:
1. Standardize data (mean=0, std=1)
2. Compute covariance matrix
3. Find eigenvectors (principal components)
4. Sort by eigenvalues (variance explained)
5. Project data onto top k components

What are principal components?
  • Directions in feature space
  • PC1: direction of most variance
  • PC2: direction of next most variance (orthogonal to PC1)
  • PC3, PC4, ... (decreasing variance)

Uses:
  1. Dimensionality reduction
     • Reduce features while preserving structure
     • Speed up ML algorithms
     • Remove noise

  2. Visualization
     • 2D/3D plots of high-dimensional data
     • Explore data structure

  3. Feature extraction
     • Create new features
     • Linear combination of original features

✅ Pros:
  • Fast (linear algebra)
  • Unsupervised (no labels needed)
  • Removes correlated features
  • Works well for visualization

❌ Cons:
  • Linear only (can miss non-linear structure)
  • Hard to interpret components
  • Sensitive to feature scaling
  • Assumes Gaussian distribution
"""

print(explanation)

## t-SNE: Non-Linear Dimensionality Reduction 🎭

In [ ]:
# t-SNE for visualization (much slower than PCA!)
print("="*60)
print("t-SNE: Non-Linear Dimensionality Reduction")
print("="*60)

print("\nComputing t-SNE (this may take 30-60 seconds)...")

# Use subset for speed
X_subset = X_digits[:500]
y_subset = y_digits[:500]

tsne = TSNE(n_components=2, random_state=42, n_iter=1000, perplexity=30)
X_tsne = tsne.fit_transform(X_subset)

print("\n✅ Done!")

In [ ]:
# Compare PCA vs t-SNE
pca_2d = PCA(n_components=2)
X_pca = pca_2d.fit_transform(X_subset)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# PCA
scatter1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y_subset,
                          cmap='tab10', s=50, alpha=0.6, edgecolors='black', linewidth=0.5)
axes[0].set_title('PCA: Linear Dimensionality Reduction')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].grid(True, alpha=0.3)

# t-SNE
scatter2 = axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_subset,
                          cmap='tab10', s=50, alpha=0.6, edgecolors='black', linewidth=0.5)
axes[1].set_title('t-SNE: Non-Linear Dimensionality Reduction')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
axes[1].grid(True, alpha=0.3)

plt.colorbar(scatter2, ax=axes, label='Digit')
plt.tight_layout()
plt.show()

print("\n✅ t-SNE creates tighter, more separated clusters!")
print("❌ But much slower and not suitable for new data projections")

In [ ]:
explanation_tsne = """
t-SNE: t-Distributed Stochastic Neighbor Embedding

Core Idea: Preserve local structure when reducing dimensions

Key Difference from PCA:
  • PCA: Preserves global structure
  • t-SNE: Preserves local structure (neighbors stay close)

How it works:
  1. Compute pairwise distances in high-D space
  2. Create probability distributions around each point
  3. Randomly initialize 2D/3D projection
  4. Minimize divergence between high-D and low-D distributions
  5. Points that are similar in high-D stay close in low-D

✅ Pros:
  • Excellent for visualization
  • Captures non-linear structure
  • Creates beautiful, interpretable plots

❌ Cons:
  • Very slow (for large datasets)
  • Can't transform new data directly
  • Sensitive to hyperparameters (perplexity)
  • Not suitable as feature engineering step

Best for: Visualization and exploration only!
"""

print(explanation_tsne)

## Real-World Example: Customer Segmentation 👥

In [ ]:
# Create customer dataset
np.random.seed(42)
n_customers = 200

# Segment 1: Budget-conscious
budget = np.random.normal(loc=[30, 2000], scale=[10, 500], size=(n_customers//2, 2))

# Segment 2: Premium customers
premium = np.random.normal(loc=[50, 8000], scale=[15, 2000], size=(n_customers//2, 2))

X_customers = np.vstack([budget, premium])

# Scale features
scaler = StandardScaler()
X_customers_scaled = scaler.fit_transform(X_customers)

print("="*60)
print("REAL-WORLD EXAMPLE: CUSTOMER SEGMENTATION")
print("="*60)

print(f"\nDataset: {n_customers} customers")
print(f"Features:")
print(f"  1. Age (years)")
print(f"  2. Annual Spending ($)")

# K-Means clustering
kmeans_customers = KMeans(n_clusters=2, random_state=42, n_init=10)
labels = kmeans_customers.fit_predict(X_customers_scaled)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original scale
colors = ['red', 'blue']
for i in range(2):
    mask = labels == i
    axes[0].scatter(X_customers[mask, 0], X_customers[mask, 1],
                   c=colors[i], label=f'Segment {i}', s=100, alpha=0.6, edgecolors='black', linewidth=0.5)
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('Annual Spending ($)')
axes[0].set_title('Customer Segments (Original Scale)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Scaled
for i in range(2):
    mask = labels == i
    axes[1].scatter(X_customers_scaled[mask, 0], X_customers_scaled[mask, 1],
                   c=colors[i], label=f'Segment {i}', s=100, alpha=0.6, edgecolors='black', linewidth=0.5)

# Plot centers
axes[1].scatter(kmeans_customers.cluster_centers_[:, 0], kmeans_customers.cluster_centers_[:, 1],
               marker='X', s=300, c='yellow', edgecolors='black', linewidth=2, label='Centers')
axes[1].set_xlabel('Age (standardized)')
axes[1].set_ylabel('Spending (standardized)')
axes[1].set_title('Customer Segments (Scaled)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze segments
print("\nSegment Analysis:")
print("-" * 60)

for segment in range(2):
    mask = labels == segment
    segment_data = X_customers[mask]
    
    print(f"\n📊 Segment {segment}: {mask.sum()} customers")
    print(f"   Age: {segment_data[:, 0].mean():.1f} years (range: {segment_data[:, 0].min():.0f}-{segment_data[:, 0].max():.0f})")
    print(f"   Spending: ${segment_data[:, 1].mean():.0f}/year (range: ${segment_data[:, 1].min():.0f}-${segment_data[:, 1].max():.0f})")
    
    if segment == 0:
        print(f"   Profile: Budget-conscious young customers")
    else:
        print(f"   Profile: Premium, high-value customers")

print(f"\n✅ Marketing strategy can be tailored per segment!")

## Clustering Algorithms Comparison 📊

In [ ]:
comparison = """
╔══════════════════╦════════════════╦═══════════════╦════════════════╗
║ Algorithm        ║ Speed          ║ Cluster Shape ║ When to Use    ║
╠══════════════════╬════════════════╬═══════════════╬════════════════╣
║ K-Means          ║ ⚡⚡⚡ Fast    ║ Spherical     ║ Baseline       ║
║                  ║                ║               ║ Large datasets ║
╠══════════════════╬════════════════╬═══════════════╬════════════════╣
║ DBSCAN           ║ ⚡⚡ Medium   ║ Any shape     ║ Non-spherical  ║
║                  ║                ║ Outliers      ║ Varying sizes  ║
╠══════════════════╬════════════════╬═══════════════╬════════════════╣
║ Hierarchical     ║ ⚡ Slow       ║ Any shape     ║ Dendrograms    ║
║ Clustering       ║                ║               ║ Small-medium   ║
╠══════════════════╬════════════════╬═══════════════╬════════════════╣
║ PCA              ║ ⚡⚡⚡ Fast    ║ N/A           ║ Visualization  ║
║ (Dimensionality) ║                ║               ║ Feature reduce ║
╠══════════════════╬════════════════╬═══════════════╬════════════════╣
║ t-SNE            ║ 🐌 Very Slow  ║ N/A           ║ Exploration    ║
║ (Visualization)  ║                ║               ║ Visualization  ║
╚══════════════════╩════════════════╩═══════════════╩════════════════╝

QUICK DECISION GUIDE:

Q: Do you know how many clusters?
  Yes → K-Means (fast, simple)
  No  → Try DBSCAN or Hierarchical

Q: Are clusters similar size and spherical?
  Yes → K-Means works well
  No  → Try DBSCAN

Q: Do you want to visualize high-D data?
  For exploration → t-SNE
  For efficiency → PCA

Q: Need to handle outliers?
  Yes → DBSCAN (marks as noise)
  No  → K-Means
"""

print(comparison)

## Evaluating Clustering Results 🎯

In [ ]:
print("="*60)
print("EVALUATING CLUSTERING QUALITY")
print("="*60)

# Create test datasets
X_test1, _ = make_blobs(n_samples=300, centers=3, n_features=2, random_state=42)
X_test2, _ = make_blobs(n_samples=300, centers=5, n_features=2, random_state=42)

# Cluster both
kmeans1 = KMeans(n_clusters=3, random_state=42)
labels1 = kmeans1.fit_predict(X_test1)
score1 = silhouette_score(X_test1, labels1)

kmeans2 = KMeans(n_clusters=5, random_state=42)
labels2 = kmeans2.fit_predict(X_test2)
score2 = silhouette_score(X_test2, labels2)

print(f"\n📊 Silhouette Score Examples:")
print(f"  Good clustering (k=3, 3 clusters): {score1:.3f}")
print(f"  Good clustering (k=5, 5 clusters): {score2:.3f}")
print(f"\n  Range: -1 to 1")
print(f"    0.7-1.0: Strong structure")
print(f"    0.5-0.7: Reasonable structure")
print(f"    0.3-0.5: Weak structure")
    < 0.3: No substantial structure")

# Visualize silhouette
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, X, labels, score in zip(axes, [X_test1, X_test2], [labels1, labels2], [score1, score2]):
    silhouette_vals = silhouette_samples(X, labels)
    
    y_lower = 10
    for i in range(len(np.unique(labels))):
        cluster_silhouette_vals = silhouette_vals[labels == i]
        cluster_silhouette_vals.sort()
        
        y_upper = y_lower + len(cluster_silhouette_vals)
        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_silhouette_vals,
                         alpha=0.7, label=f'Cluster {i}')
        y_lower = y_upper + 10
    
    ax.axvline(x=score, color='red', linestyle='--', linewidth=2, label=f'Mean: {score:.3f}')
    ax.set_xlabel('Silhouette Coefficient')
    ax.set_ylabel('Cluster')
    ax.set_title(f'Silhouette Plot (Score: {score:.3f})')
    ax.legend(fontsize=9, loc='best')

plt.tight_layout()
plt.show()

## Key Takeaways 🎯

✅ **K-Means:** Fast, simple baseline. Use when you know number of clusters

✅ **DBSCAN:** Handles arbitrary shapes, finds outliers. Use when clusters not spherical

✅ **PCA:** Linear dimensionality reduction. Use for visualization and feature compression

✅ **t-SNE:** Non-linear, beautiful visualizations. Use for exploration, NOT production

✅ **Elbow Method:** Find optimal clusters by looking for 'elbow' in inertia plot

✅ **Silhouette Score:** Evaluate cluster quality (-1 to 1, higher is better)

✅ Always scale features before clustering

✅ No single "best" algorithm - choose based on data and problem

## Practice Exercise 💪

1. **Clustering:** Apply K-Means, DBSCAN, and Hierarchical to same dataset - compare results
2. **Optimal k:** Use elbow method and silhouette score on different k values
3. **Dimensionality Reduction:** Use PCA and t-SNE on digits dataset, visualize
4. **Customer Segmentation:** Create your own customer dataset and find segments
5. **Evaluation:** Compare silhouette scores for different clustering results

Challenge: Find the "true" number of clusters in a synthetic dataset using multiple methods!

## Next Steps 🚀

→ **Real Projects:** Apply clustering/dimensionality reduction to real data

→ **Advanced:** Mixture Models, Spectral Clustering, Autoencoders

→ **Time Series:** Forecasting, anomaly detection in sequences

→ **Production:** Deploy models, monitoring, retraining pipelines

→ **Next Level:** NLP, Computer Vision, Deep Learning specialization